# ■ データの前処理（medprep）― 第3回 演習

## □ 日本腎・血液浄化AI学会  学術委員会作成&nbsp;&nbsp;&nbsp;  made on September 18th, 2026

### ◇ MIT License&nbsp;&nbsp;&nbsp;&nbsp;Version 1.0 &nbsp;&nbsp;(Colab / Windows / Mac 共通版)

第1回・第2回で作った環境の上で、**解析を始める前の仕事**を扱う。
医学研究のデータは、そのままでは解析にかけられない。日付が 8 通りの書き方で混ざり、
欠損が `999` や「未測定」と書かれ、単位が途中で変わり、測定法が変わって値が段差になる。
**この回で覚えるのは、それらを見つけて記録に残す手順**である。

### ◇ この演習の一番大事な考え方

> **全自動とは「決定の自動化」ではなく「決定の明示化」である。**

`medprep` は 1 行で最後まで走る。しかし**そこで下した判断はすべて外に出る**。
「この列は ID とみなした」「この値は欠損コードとみなした」「この症例は除外した」——
その理由が `schema.yaml` と「人の確認が要る事項」に残り、**人が直して再実行できる**。

**確認事項が空で返ってきたら、まずそちらを疑うこと。** 医学データで所見ゼロはまず無い。

### ◇ このコードの構成

第１部  環境設定: 最初に**一回実行**。新たなデータ分析時は再実行<br>
   　1) Google Drive にマウント: **Colaboratory** なら必ず実行<br>
   　2) 環境変数の一括設定: 最初に**一回実行**<br>
   　3) Google Colaboratory への Library のインストール: **Colaboratory** なら必ず実行<br>
   　4) データの入力: 新規のデータのたびに実行<br>

第２部  演習（上から順に実行する）<br>
   　演習① まず全自動で走らせる（`mp.autoprep` 1 行）<br>
   　演習② 機械の判断を検分する（`schema` と監査）<br>
   　演習③ 欠損と外れ値 ―― **消してよい場合と消してはいけない場合**<br>
   　演習④ Table 1 と群間比較 ―― 検定の自動選択、効果量、p 値の罠<br>
   　演習⑤ 生存時間分析 ―― KM → log-rank → Cox → 比例ハザードの確認<br>
   　演習⑥ `schema.yaml` を直して再実行し、モデルに渡す<br>

第３部  発展事項<br>

### ◇ 分析するデータの準備

1. Excel（または CSV）に、**1列が1つの変数、1行が1症例**となるように入力する。
2. **目的変数の位置は自由**。名前で指定する（環境変数セルの `OUTCOME`）。
3. **記号や単位が混ざっていてもよい。** `<0.1`、`999`、「未測定」、全角数字はそのまま入れてよい。
   何をどう解釈したかは必ず記録に残る。
4. 不明な値は**空欄**にする。**空欄を 0 で埋めてはならない。**
5. 日付は書き方が混ざっていてよい（和暦・Excel シリアル値・全角・時刻付き）。

### ◇ 使い方

◆ **Colaboratory**: このコードを Google Drive にアップロードして開き、上から順に実行する。<br>
◆ **Windows / Mac**: `jl` で JupyterLab を立ち上げ、**code** にこのコードを置いて開く。
分析する Excel / CSV は **data** に入れる。

計算結果は Colab なら `MyDrive/AI/lab_output/Preprocessing`、PC なら `~/lab_output/Preprocessing`
の中に `run1`, `run2`, ... という新しいフォルダが作られて保存される。
**過去の結果が上書きされることはない。** 実行の記録は **log** フォルダの `history.csv` に貯まる。

### ◇ 自分のデータが無い場合

環境変数セルの `USE_DEMO_DATA = True` のままにしておくと、
**演習用の合成透析コホート**（600 例・実データは一切含まない）が使われる。
上に挙げた汚れがすべて意図的に仕込んである。

# ◆ 1. 環境構築 (最初に一回だけ実行する)

## 1) Google Colaboratoryを使う場合このNotebookをGoogle Driveにマウントする<br>（ローカルPCでは実行してもエラーにはならない）

In [ ]:
# このセルは Google Colaboratory で Google Drive をマウントする。
# ローカル（Windows / Mac）では IN_COLAB が False になり、マウントはスキップされる。
import sys

# 実行環境の判定（このセルを単独で実行できるよう、ここでも判定しておく。第1部2)でも再判定する）
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('◇ ローカル環境のため Google Drive のマウントはスキップした')

print(f"Python version: {sys.version.split()[0]}")

## 2) 環境変数の一括設定&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;ここを編集することで解析の対象と設定を変えられる<br>（環境変数を書き換えたらこのセルを再実行）

In [ ]:
# ============================================================================
# ■ 環境変数の一括設定
#   ここを書き換えて再実行するだけで、以降のすべてに反映される。
# ============================================================================
import os

# 実行環境の判定（Colab / ローカル：Windows・Mac 共通）
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ----- 保存先（第1回・第2回で作ったフォルダの規約をそのまま使う） -----
#   ~/lab          … uv の環境とプログラム        PROJECT_DIR
#   ~/lab_work     … code / data / log            WORK_DIR
#   ~/lab_output   … 計算結果 run1, run2, ...     OUTPUT_DIR
#   どれも '' のままなら自動で決まるので、触る必要はない。
PROJECT_DIR = ''
WORK_DIR    = ''
OUTPUT_DIR  = ''
COLAB_BASE  = '/content/drive/MyDrive/AI'

# ----- このノートブックの名前（実行の記録に残す） -----
NOTEBOOK_NAME = 'Preprocessing_Ver1_0'
RUN_BY        = '手動（JupyterLab / Colab）'
PROJECT_NAME  = 'Preprocessing'      # lab_output/{ここ}/run{N}/ に保存される

# ----- 読み込むデータ -----
#   USE_DEMO_DATA = True  → 演習用の合成データを使う（実データは含まない）
#   LOCAL_DATA_PATH にパスを書けばそのファイルを読む。空なら GUI で選ぶ。
USE_DEMO_DATA   = True
LOCAL_DATA_PATH = ''

# ----- 列の指定（自分のデータを使うときはここを書き換える） -----
#   ★ここで指定した役割は、機械の推定より強い。★
ID_COL  = '仮名ID'        # 患者を識別する列。分割のときに同じ患者を両側に入れないために使う
GROUP   = '施設'          # 群間比較・グループ分割に使う列（無ければ None）
DATE_COL = '検体採取日'    # 測定法変更の段差を調べるための日付列（無ければ None）

# 生存時間：観察開始日 / イベント発生日 / 打ち切り日 の 3 列（無ければ None）
SURVIVAL_DATES = ('観察開始年月日', 'event発生年月日', '観察打ち切り年月日')

# 目的変数（回帰・分類をするとき）。演習⑥で作るのでここでは None。
OUTCOME = None
TASK    = None            # 'regression' / 'classification' / 'survival'

# ----- 分割 -----
TEST_SIZE    = 0.25
RANDOM_STATE = 0

# ----- レポート -----
#   ★既定では症例レベルの値（仮名 ID など）をレポートに出さない。★
#   レポートは HTML 1 枚でメールに乗る。必要なときだけ True にすること。
SHOW_VALUES = False

print('◇ 環境変数を設定した')
print(f'   IN_COLAB = {IN_COLAB}  （True=Colab / False=ローカルPC）')
print(f'   PROJECT_NAME = {PROJECT_NAME}')
print(f'   USE_DEMO_DATA = {USE_DEMO_DATA}')

## 3) Google Colaboratoryへのライブラリのインストール<br>（ローカルPCで実行してもスキップされる）

In [ ]:
# ライブラリのインストール
#   Colab        : 実行のたびに環境が初期化されるため、毎回 pip で導入する。
#   ローカル(uv) : 事前に uv で導入済みのため、このセルはスキップされる。
#
# ★このノートブックが必要とする medprep の版★
#   古い medprep が入っていると、あとのセルが AttributeError で止まる。
#   ここで版を確かめて、足りなければ**理由を言って止める**。
REQUIRED_MEDPREP = (0, 2, 0)

if IN_COLAB:
    # -U（更新）と --no-cache-dir を付ける。付けないと、同じ版番号のまま
    # 中身だけ更新された medprep を pip が「導入済み」とみなして取りに行かない。
    !pip install -q -U --no-cache-dir "medprep @ git+https://github.com/kiwindow/medprep"
    print('◇ Colab: medprep を導入した')
else:
    print('◇ ローカル（uv）環境: pip install はスキップした')
    print('   ※ 未導入・古い場合は、ターミナルで次を実行すること:')
    print('      uv add --upgrade "medprep @ git+https://github.com/kiwindow/medprep"')

print('')

import warnings

import matplotlib_fontja  # noqa: F401   日本語フォント

import medprep as mp

# 警告表示の抑制
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# 環境変数セルで決めた保存先を medprep に渡す
mp.paths.PROJECT_DIR = PROJECT_DIR
mp.paths.WORK_DIR    = WORK_DIR
mp.paths.OUTPUT_DIR  = OUTPUT_DIR
mp.paths.COLAB_BASE  = COLAB_BASE

def _ver(text):
    return tuple(int(x) for x in str(text).split('.')[:3])


if _ver(mp.__version__) < REQUIRED_MEDPREP:
    need = '.'.join(map(str, REQUIRED_MEDPREP))
    for line in [
        f'★medprep が古い（入っているのは {mp.__version__}、必要なのは {need} 以上）★',
        '',
        '  Colab   : このセルをもう一度実行したうえで、',
        '            「ランタイム → セッションを再起動する」を実行し、上から流し直すこと。',
        '            ★一度読み込まれた古い medprep は、再起動しないと入れ替わらない。★',
        '',
        '  ローカル : uv add --upgrade "medprep @ git+https://github.com/kiwindow/medprep"',
    ]:
        print(line)
    raise RuntimeError(f'medprep が古い（{mp.__version__} < {need}）。上の指示に従うこと')

print(f'◇ medprep {mp.__version__}')
print(f'   結果の保存先 : {mp.paths.resolve_output_dir()}')
print(f'   実行の記録   : {mp.paths.resolve_log_dir()}')

## 4) **データの入力** (.xlsx, .xls, .csv) &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;新たなデータを使う時にも実行

- `USE_DEMO_DATA = True` なら、演習用の合成データが `data` フォルダに作られる。
- 自分のデータを使うときは `USE_DEMO_DATA = False` にして、
  `LOCAL_DATA_PATH` にパスを書くか、空のままにして GUI で選ぶ。
- **記号・単位・全角・和暦が混ざっていてよい。** どう解釈したかは必ず記録に残る。

In [ ]:
import pandas as pd

DATA_DIR = os.path.join(mp.paths.resolve_work_dir(), 'data')
os.makedirs(DATA_DIR, exist_ok=True)

if USE_DEMO_DATA:
    # 演習用の合成透析コホート（実データは一切含まない）
    data_path = os.path.join(DATA_DIR, 'synthetic_dialysis_cohort.xlsx')
    if not os.path.exists(data_path):
        mp.demo.save(data_path)
        print(f'◇ 演習用の合成データを作った: {data_path}')
elif LOCAL_DATA_PATH:
    data_path = os.path.expanduser(LOCAL_DATA_PATH)
elif IN_COLAB:
    from google.colab import files
    up = files.upload()
    data_path = os.path.join(DATA_DIR, next(iter(up)))
    with open(data_path, 'wb') as f:
        f.write(next(iter(up.values())))
else:
    from ipyfilechooser import FileChooser
    fc = FileChooser(DATA_DIR)
    display(fc)
    data_path = fc.selected
    if data_path is None:
        print('★ファイルを選んでから、このセルをもう一度実行すること★')

# ★読み込みで推測したこと（文字コード・見出し行）は必ず記録に残る★
df = mp.read_any(data_path)
for note in df.attrs.get('medprep_read', []):
    print(' ・', note)

print('')
print(f'◇ {df.shape[0]} 行 × {df.shape[1]} 列')
df.head()

# ◆ 2. 演習① まず全自動で走らせる

読み込みから前処理済みの行列とレポートまでを 1 行で通す。

**ここで見るのは結果ではなく、「機械が何を判断したか」である。**
段ごとの成否と、人の確認が要る事項が最後にまとめて出る。

In [ ]:
rep = mp.autoprep(
    df,
    group=GROUP, id_col=ID_COL, date_col=DATE_COL,
    survival_dates=SURVIVAL_DATES,
    outcome=OUTCOME, task=TASK,
    test_size=TEST_SIZE, seed=RANDOM_STATE,
    show_values=SHOW_VALUES,
    save=True, method=PROJECT_NAME,      # run{N}/ に全出力を保存する
)

In [ ]:
# 段ごとの成否と、人の確認が要る事項
rep.show()

## レポートを開く

`run{N}/report/prep_report.html` を開くと、監査・列の役割・欠損・外れ値・Table 1・
図が **HTML 1 枚**にまとまっている。図は埋め込みなので、そのままメールに乗る。

**冒頭に致命的な所見が並んでいる。** 10 枚目にある警告は、読まれないのと同じだからである。

In [ ]:
from IPython.display import IFrame, display

html_path = rep.run.file('report', 'prep_report.html')
print(html_path)

try:
    display(IFrame(os.path.relpath(html_path), width='100%', height=560))
except Exception:
    print('※ ここに表示できない場合は、上のパスのファイルをブラウザで開くこと')

# ◆ 3. 演習② 機械の判断を検分する

### ここが第3回の山場である。

`autoprep` が下した判断には**必ず理由が付いている**。
その理由を読み、納得できなければ直す。これが「決定の明示化」の意味である。

## 1) 列の役割（schema）

In [ ]:
print(rep.schema.report())

In [ ]:
# 表で見る。「判断の根拠」の列を必ず読むこと。
rep.schema.to_frame()

**確かめること**

- `drop` になった列は、本当に捨ててよいか（`id` / `duplicate_of` / `high_cardinality`）
- `numeric` になった列に、数値でないものが混ざっていないか
- `datetime` の列が、日と月を取り違えていないか

直したいときは `schema.yaml` を編集して読み直す。

In [ ]:
yaml_path = rep.run.file('model', 'schema.yaml')
print(yaml_path)
print()
with open(yaml_path, encoding='utf-8') as f:
    print(f.read()[:1500], '...')

## 2) 品質監査 ―― このまま解析してよいかを問う

In [ ]:
# ★audit はデータを直さない。報告するだけである。★
rep.audit.show()

In [ ]:
# 表にして、致命的な所見から順に見る
rep.audit.to_frame()

### 測定法の変更に注目する

合成データには **2020 年 4 月の ALP 測定法変更（JSCC 法 → IFCC 法）** が仕込んである。
値がおよそ 1/3 になるが、**病態は何も変わっていない。**

この段差を見落とすと「2020 年以降、患者の ALP が下がった」という誤った結論になる。
検定でも例外でも捕まらない。**日付で切って中央値を比べて初めて見える。**

In [ ]:
steps = mp.method_change_steps(df, date_col=DATE_COL)
steps

# ◆ 4. 演習③ 欠損と外れ値

### 「測っていない」ことが情報である

重症だから測った／軽症だから測らなかった、という欠損は予後と結びついている。
欠損を黙って中央値で埋めると、**その情報を捨てたうえに分布を歪める。**

In [ ]:
print(rep.missing.report())

In [ ]:
# 欠損の地図。縦の筋が揃っていれば、同じ検査がまとめて行われなかったということ。
rep.missing.plot(rep.df_clean, kind='matrix')

### 欠損は偏っているか

欠損が完全にランダム（MCAR）なら、欠損フラグはどの列とも関連しないはずである。
関連があれば単純補完は分布を歪める。**「どの列の欠損がどの列と結びついているか」を名指しする。**

In [ ]:
rep.mcar

## 外れ値 ―― 消してよい場合と消してはいけない場合

| | 例 | 扱い |
|---|---|---|
| **入力ミス** | Hb 0 g/dL、年齢 250 歳 | **生理学的にあり得ない。** 辞書の範囲で NaN にする |
| **本物の外れ値** | Alb 1.8 g/dL、CRP 25 mg/dL | **消してはならない。** 医学ではこれが重要な症例でありうる |

この区別は統計では付かない。**医学領域辞書（`dict/ranges_ja.yaml`）が担う。**
`medprep` は既定で外れ値を削除しない。検出して報告するだけである。

In [ ]:
print(rep.outliers.report())

In [ ]:
rep.outliers.table

# ◆ 5. 演習④ Table 1 と群間比較

検定は自動で選ぶ。ただし**選んだ理由と効果量を必ず添える。**

In [ ]:
print(rep.table1.report())

### 検定の選択理由・効果量・多重比較の補正

- 連続変数は正規性の判定に従って 平均(SD) か 中央値[Q1,Q3] を選ぶ
- 3 群以上なら事後比較（Tukey HSD / Dunn）まで出す
- **全列に p を出すので、BH の q 値を必ず併記する**
- **群間の「偏り」は p ではなく SMD で見る**（p は例数で動く）

In [ ]:
rep.table1.comparison.to_frame()[['項目', '検定', 'p', 'q(BH)', '効果量', 'SMD', '判定の根拠']]

## 管理目標の達成率 ―― 開区間と閉区間を取り違えない

JSDT の管理目標には**境界値を含むものと含まないものがある**。
`<` と `<=` の取り違えは、達成率を数％動かす。**論文の結論が変わる。**

In [ ]:
rep.achievement

In [ ]:
# 境界値ちょうどの症例が何例あるか。ここが取り違えの影響を受ける。
clean = rep.df_clean
for key, col in [('P', '無機リン(P)'), ('cCa', '補正Ca'), ('Hb', '末梢血｜血色素量(Hb)')]:
    if col not in clean.columns:
        continue
    spec = mp.load_dict()['items'][key]
    tg = spec['target']
    v = pd.to_numeric(clean[col], errors='coerce')
    ok = v.notna()
    right = mp.in_target(v, tg)
    wrong = mp.in_target(v, {**tg, 'high_inclusive': True, 'low_inclusive': True})
    a, b = right[ok].mean(), wrong[ok].mean()
    print(f"{spec['name_ja']:<14s} 正しく開区間 {a:6.1%} / 誤って閉区間 {b:6.1%}"
          f"  → 差 {b - a:+.1%}（{int((wrong[ok] != right[ok]).sum())} 例）")

# ◆ 6. 演習⑤ 生存時間分析

4 列の日付（ID / 観察開始日 / イベント発生日 / 打ち切り日）から `(duration, event)` を作る。

**矛盾した症例は勝手に解釈しない。除外して、理由を表に残す。**

In [ ]:
sf = rep.survival
print(sf.report())
print()
print('除外された症例:')
sf.excluded

In [ ]:
import numpy as np

d = sf.data.rename(columns={
    'アルブミン(Alb)': 'Alb', '末梢血｜血色素量(Hb)': 'Hb',
    'C反応性蛋白(CRP)定量': 'CRP', '無機リン(P)': 'P', '透析歴_月': 'vintage'})
d['logCRP'] = np.log(pd.to_numeric(d['CRP'], errors='coerce') + 0.1)
d['低Alb'] = np.where(pd.to_numeric(d['Alb'], errors='coerce') < 3.5, 'Alb<3.5', 'Alb≧3.5')

surv = mp.Survival(d, unit='years')
km = surv.km(by='低Alb', title='アルブミン値による生存曲線',
             save=rep.run.file('figure', 'km_alb.png'))
print(km.report())

In [ ]:
print(surv.logrank(by='低Alb').report())

## Cox 比例ハザード回帰 ―― **黙って減る n を見張る**

Cox は共変量に欠測がある症例を**黙って落とす**。
600 例で始めたはずが 400 例で推定されている、ということが起こる。
`medprep` は**何例落ちたかを必ず報告する。**

あわせて **EPV（イベント数 ÷ 共変量の数）** を見る。10 を下回れば推定が不安定になる。

In [ ]:
cov = [c for c in ['年齢', 'Alb', 'Hb', 'logCRP', '糖尿病', 'vintage'] if c in d.columns]
cox = surv.cox(covariates=cov)
print(cox.report())

In [ ]:
surv.forest(cox, save=rep.run.file('figure', 'cox_forest.png'))

### 比例ハザード仮定の確認

Cox は「ハザード比が時間によらず一定」と仮定している。
Schoenfeld 残差の検定が有意なら**その仮定は成り立っていない**。
そのときは層別化するか、比例ハザードを仮定しない指標（RMST）を使う。

In [ ]:
surv.schoenfeld_plot(cox, save=rep.run.file('figure', 'cox_schoenfeld.png'))
print('PH 違反:', cox.ph_violations or 'なし')

### 合成データの「真の係数」と照合する

演習用の合成データは、**分かっている係数から作ってある**。
推定した 95% 信頼区間が真値を含むかを確かめると、手順が正しいことを自分で検証できる。
（自分のデータでは真値は分からない。だからこそ合成データで手順を確かめておく。）

In [ ]:
truth = mp.demo.TRUE_COEFFICIENTS
m = cox.model
chk = pd.DataFrame({
    '真の係数': pd.Series(truth),
    '推定係数': m.params_,
    '95%CI下限': m.confidence_intervals_.iloc[:, 0],
    '95%CI上限': m.confidence_intervals_.iloc[:, 1]}).dropna()
chk['CIに真値を含む'] = ((chk['95%CI下限'] <= chk['真の係数'])
                    & (chk['真の係数'] <= chk['95%CI上限']))
print(f"{int(chk['CIに真値を含む'].sum())}/{len(chk)} の変数で 95%CI が真値を含む")
chk.round(3)

# ◆ 7. 演習⑥ 目的変数を作り、モデルに渡す

### リークを構造的に不可能にする

「前処理をしてから分割する」と、テストデータの情報が前処理に混ざる（**リーク**）。
中央値も、スケーラも、カテゴリの一覧も、**train だけで決めなければならない。**

`medprep` では `fit` を test に呼ぶと**例外で止まる**。気をつける話ではなく、**できない構造にする。**

In [ ]:
# 目的変数：1 年以内のイベント発生
#   ★1 年経たずに打ち切られた症例は「1 年以内に起きたか」を判定できない。★
#     その症例は除く。除いた数は必ず報告される。目的変数を補完してはならない。
d1 = d.copy()
d1['1年以内イベント'] = np.where(d1['event'] == 1,
                             (d1['duration'] <= 1.0).astype(float), 0.0)
d1.loc[(d1['event'] == 0) & (d1['duration'] < 1.0), '1年以内イベント'] = np.nan

rep2 = mp.autoprep(
    d1,
    outcome='1年以内イベント', task='classification',
    group=GROUP, id_col=ID_COL,
    test_size=TEST_SIZE, seed=RANDOM_STATE,
    columns=[c for c in d1.columns if c not in ('duration', 'event', '低Alb', 'CRP')],
    save=True, method=PROJECT_NAME,
)

In [ ]:
rep2.show()

In [ ]:
# リークの検査。★通ったことの証拠として残せる表である。★
mp.leak_check(rep2.pipeline, rep2.split.train, rep2.split.test)

In [ ]:
# test に fit しようとすると止まる
try:
    mp.Preprocessor(rep2.schema).fit(rep2.split.test)
except mp.LeakageError as e:
    print('✗', str(e).split('。')[0], '。')

## 前処理済みの行列をそのままモデルに渡す

`rep2.X_train` は **列名の付いた DataFrame** である。
`get_dummies` で作った列の名前も残っているので、**係数を読める。**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model = LogisticRegression(max_iter=2000).fit(rep2.X_train, rep2.y_train)
auc_tr = roc_auc_score(rep2.y_train, model.predict_proba(rep2.X_train)[:, 1])
auc_te = roc_auc_score(rep2.y_test, model.predict_proba(rep2.X_test)[:, 1])
print(f'ロジスティック回帰  train AUC = {auc_tr:.3f} / test AUC = {auc_te:.3f}')

coef = (pd.Series(model.coef_[0], index=rep2.X_train.columns)
        .sort_values(key=abs, ascending=False).head(10))
coef.round(3)

### 再現に必要なもの

| ファイル | 中身 |
|---|---|
| `model/schema.yaml` | **列の役割と、そう判断した理由。これ 1 枚で前処理を再現できる** |
| `model/pipeline.pkl` | fit 済みの前処理（sklearn 互換） |
| `table/run_info.json` | run 番号と保存先（D&D アプリが読む） |
| `log/history.csv` | どのデータを、どのコードで、どう解析したかの一覧 |

In [ ]:
print(rep2.run.report_text())
print()
for p in rep2.saved:
    print(' ', p)

# ◆ 8. 発展事項

## 全自動にしてはいけないもの

このパッケージの値打ちは「何を自動化したか」と同じくらい、**「何を自動化しなかったか」**で決まる。
以下は**警告を出して人に返す**。自動で処理しない。

| 項目 | 理由 |
|---|---|
| **外れ値の自動削除** | 医学では外れ値こそが重要な症例（劇症型、稀な合併症）でありうる |
| **目的変数の補完** | 補完した目的変数で学習した結果は解釈できない。除外して、除外数を報告する |
| **ステップワイズ変数選択** | 逐次選択は推定量に偏りを与え、信頼区間と p 値が無効になる |
| **p 値による Table 1 の解釈** | 群間の偏りは p ではなく SMD で見る。p は例数で動く |
| **多重比較の無補正** | 全列に p を出すので、BH の q 値を必ず併記する |
| **PH 仮定違反時の Cox の黙認** | Schoenfeld 検定が有意なら警告し、層別化か RMST を勧める |
| **EPV < 10 での多変量 Cox** | 警告を出す |
| **施設・時期をそのまま特徴量にすること** | リークの温床。投入するかを人に問う |
| **欠損の指示変数を黙って落とすこと** | 「測っていない」ことが予後情報である |

> 受講者が持ち帰るべき原理は、
> **「自動化できる部分と、医学的判断が必要な部分の境界を知ること」**である。

## この先（第4回以降）

- **MICE（多重代入）** ―― 欠損が MAR のときの正しい扱い。補完の不確実性を推定に反映させる
- **target encoding** ―― 水準の多いカテゴリ。**fold の中で fit しないとリークする**
- **競合リスク** ―― 死亡と移植のように、片方が起きるともう片方が観察できない場合
- **時系列** ―― 横持ちの検査値を時系列として扱う（第4回）

## 自分のデータで試す

環境変数セルに戻り、

1. `USE_DEMO_DATA = False` にする
2. `ID_COL` / `GROUP` / `DATE_COL` / `SURVIVAL_DATES` を自分の列名に書き換える
3. このノートブックを上から実行する

**同じコードが自分のデータでも動く。** そこが第3回の山場である。